In [ ]:
from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


def Zip(*a):
    return np.stack([np.array(a, "O") for a in a], 1)


def Flat(a):
    return a.ravel().tolist()


def Unzip(a, n):
    return np.array(a, "O").reshape(-1, n)


def Rand(a):
    return (np.random.rand(len(a)) * a).astype(int)


def Slice(a, i, j):
    return a.split(Flat(np.diff(Zip(i, j).ravel(), prepend=0, append=len(a))))[1::2]


class Dat:
    m = {}
    d = {"true": 0, "false": 1}

    def __init__(s, a, b=8):
        x = []
        l = []
        y = []
        for a in open(f"cluewsc2020/{a}.json"):
            a = loads(a)
            X = []
            L = []
            for t in a["text"], a["target"]["span1_text"], a["target"]["span2_text"]:
                if not b:
                    t = [t for t in t if t in s.m]
                    if not t:
                        break
                X.extend(s.m.setdefault(t, len(s.m)) for t in t)
                L.append(len(t))
            else:
                x.extend(X)
                l.extend(L)
                y.append(int(s.d[a["label"]]))
        s.x = torch.tensor(x, device="cuda")
        s.l = np.array(l)
        s.t = np.r_[0, s.l[:-1]].cumsum()
        s.y = torch.tensor(y, dtype=torch.float, device="cuda")[:, None]
        s.b = b

    def __len__(s):
        return len(s.y) // s.b if s.b else 1

    def __iter__(s):
        if s.b:
            for t in range(len(s)):
                t = np.random.choice(len(s.y), s.b, 0)
                t.sort()
                T = (t[:, None] * 3 + [0, 1, 2]).ravel()
                a = s.l[T]
                l = 1 + Rand(a)
                i = s.t[T] + Rand(a - l)
                yield [torch.cat(Slice(s.x, i, i + l)), l], s.y[torch.tensor(t, device="cuda")]
        else:
            yield [s.x, s.l], s.y


class Embedding(Embedding):
    def forward(s, x):
        return super().forward(x[0]), x[1]


class BatchNorm1d(BatchNorm1d):
    def forward(s, x):
        return super().forward(x) if len(x) > 1 else x


class Red(Sequential):
    def forward(s, x):
        x, l = x
        while len(x) > len(l):
            t = Rand(l) & 4294967294
            l = Zip(t, l & 1, (l & 4294967294) - t)
            x = Unzip(x.split(Flat(l)), 3)
            l[:, ::2] >>= 1
            x[:, ::2] = Unzip(super().forward(torch.cat(Flat(x[:, ::2])).view(-1, x[0, 0].size(1) * 2)).split(Flat(l[:, ::2])), 2)
            x = torch.cat(Flat(x))
            l = l.sum(1)
        return x


def accuracy(y, t):
    return int((y.signbit() != t).sum()) / len(y)

In [ ]:
m = Learner(
    DataLoaders(Dat("train"), Dat("dev", 0)),
    Sequential(
        Embedding(len(Dat.m), 64),
        Red(Linear(2 * 64, 64), BatchNorm1d(64, track_running_stats=0), Swish()),
        View(-1, 3 * 64),
        Linear(3 * 64, 64),
        BatchNorm1d(64, track_running_stats=0),
        Swish(),
        Linear(64, 1),
    ).cuda(),
    BCEWithLogitsLoss(),
    metrics=acc,
    cbs=ShowGraphCallback,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(500, .001, 9, 999, .01)

In [ ]:
m.recorder.values[-1]

In [ ]:
m.save('cluewsc2020')